# Install Libraries

In [ ]:
!pip install python-dotenv

# Get ENV variable

In [ ]:
# For Local DEV
from dotenv import load_dotenv
load_dotenv()

# Get Variable
import os
GCS_PROJECT= os.getenv('GCS_PROJECT')
GCS_BUCKET= os.getenv('GCS_BUCKET')
GCS_FILE_PATH= os.getenv('GCS_FILE_PATH')
BQ_TARGET_PROJECT= os.getenv('BQ_TARGET_PROJECT')
BQ_TARGET_SCHEMA= os.getenv('BQ_TARGET_SCHEMA')
BQ_TARGET_TABLE= os.getenv('BQ_TARGET_TABLE')

# Modify ENV
LOCAL_PATH= os.path.split(GCS_FILE_PATH)[-1]

# Get Raw File

In [ ]:
from google.cloud import storage

def download( gcs_bucket, bucket_file, local_file):
    if os.getenv('LOCAL_TEST') :  os.environ["GOOGLE_APPLICATION_CREDENTIALS"]= 'key.json'
    client = storage.Client()
    bucket = client.bucket(gcs_bucket)
    blob = bucket.blob(bucket_file)
    blob.download_to_filename(local_file)
    if os.getenv('LOCAL_TEST') :  _= os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS")
    
download(gcs_bucket= GCS_BUCKET, bucket_file= GCS_FILE_PATH, local_file= LOCAL_PATH)

# Data Processing

## Read File

In [ ]:
with open(LOCAL_PATH, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\r") 
             for line in f.read().split("\n")
             if line.strip() != ""]

# Get header
header= lines[0]
# Skip header and blank lines
data_lines= lines[1:]

print(f"Total lines: {len(lines)}")
print(f"Data rows  : {len(data_lines)}")
print(f"Header : {header}")
print("Data lines :")
for line in data_lines[:3]:
    print(line)

## Split lines into columns

In [ ]:
def split_line_to_columns(line: str) -> list:
    tokens = line.split(",")

    # integer_col may contain thousands separators such as "261,180"
    idx = 1

    while (
        idx < len(tokens)
        and tokens[idx - 1].strip().isdigit()
        and tokens[idx].strip().isdigit()
        and len(tokens[idx].strip()) == 3
    ):
        idx += 1

    integer_col = ",".join(tokens[0:idx]).strip().replace(',','')
    decimal_col = tokens[idx].strip() if idx < len(tokens) else ""
    timestamp_col = tokens[idx + 1].strip() if idx + 1 < len(tokens) else ""
    boolean_col = tokens[idx + 2].strip() if idx + 2 < len(tokens) else ""
    holiday_name = ",".join(tokens[idx + 3:]).strip() if idx + 3 < len(tokens) else ""

    return [
        integer_col,
        decimal_col,
        timestamp_col,
        boolean_col,
        holiday_name,
    ]

rows = [split_line_to_columns(line) for line in data_lines]
rows[:2]

## Push into DataFrame

In [ ]:
import pandas as pd

df= pd.DataFrame(rows, columns= header.split(','))
df= df.reset_index().rename(columns= {'index' : 'row_id'})
df['row_id'] += 1
df.head(3)

## Cleansing Numeric

In [ ]:
# Decimal Columns
df['decimal_col']= pd.to_numeric(df['decimal_col'], errors='coerce')

# INT Columns
df['integer_col']= pd.to_numeric(df['integer_col'], errors='coerce').astype('Int64')

## Cleansing Boolean

In [ ]:
def clean_boolean_value(raw):
    value = raw.lower()
    if (not value)|(value == "-"): return None
    if value in ["true", "yes", "ok", "1"]: return True
    return False

df['boolean_col']= df['boolean_col'].apply(clean_boolean_value)

## Cleansing DT

In [ ]:
from datetime import datetime

def clean_timestamp_value(raw):
    value = raw.lower()
    if (not value)|(value == "-"): return None
    # Pattern
    DT_PATTERN = [    
                    "%Y-%m-%d %H:%M:%S", "%Y%m%d%H%M%S",
                    "%d/%m/%Y", "%y-%b-%d",
                    "%Y-%m-%d", "%Y%m%d",
                 ]
    # Find matched pattern
    for fmt in DT_PATTERN :
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    return None

df['timestamp_col']= df['timestamp_col'].apply(clean_timestamp_value)

## Extract Holiday

In [ ]:
def extract_holiday_value(raw):
    value= raw.strip('"').strip("'").strip().lower()
    if (not value)|(value == "-"): return None
    # Pattern
    HOLIDAY_PATTERNS = [
                            ("Makha Bucha Day", ["makha bucha"]),
                            ("Labour Day", ["labour day"]),
                            ("Chulalongkorn Day", ["chulalongkorn day"]),
                            ("King's Birthday", ["king's birthday"]),
                            ("King Bhumibol Memorial Day", ["king bhumibol", "memorial day"]),
                            ("Songkran Festival", ["songkran festival"]),
                            ("Constitution Day", ["constitution day"]),
                            ("Royal Ploughing Ceremony", ["royal ploughing ceremony"]),
                            ("Queen's Birthday", ["queen's birthday"]),
                            ("New Year's Day", ["new year's day"]),
                            ("Christmas Day", ["christmas day"]),
                            ("Buddhist Lent", ["buddhist lent"]),
                        ]
    # Find best match (based on position)
    best_name, best_pos = None, None
    for name, keywords in HOLIDAY_PATTERNS:
        # Meet all keywords
        if all(keyword in value for keyword in keywords):
            # First found
            pos = min(value.find(keyword) for keyword in keywords)
            # Assign
            if best_pos is None or pos < best_pos :
                best_name, best_pos = name, pos
    return best_name

df['holiday_name'] = df['holiday_name'].apply(extract_holiday_value)

## Add Run time

In [ ]:
df["business_datetime"] = pd.Timestamp.now()
df["created_datetime"] = df["business_datetime"] + pd.Timedelta(hours= -7)

df = df[[
    "row_id",
    "integer_col",
    "decimal_col",
    "timestamp_col",
    "boolean_col",
    "holiday_name",
    "business_datetime",
    "created_datetime",
]]

df.head()

# GBQ Loading Function

## Prep Schema

In [ ]:
schema_setup= [
  {
    "name": "row_id",
    "mode": "NULLABLE",
    "type": "INTEGER",
    "description": "",
    "fields": []
  },
  {
    "name": "integer_col",
    "mode": "NULLABLE",
    "type": "INTEGER",
    "description": "",
    "fields": []
  },
  {
    "name": "decimal_col",
    "mode": "NULLABLE",
    "type": "FLOAT64",
    "description": "",
    "fields": []
  },
  {
    "name": "timestamp_col",
    "mode": "NULLABLE",
    "type": "TIMESTAMP",
    "description": "",
    "fields": []
  },
  {
    "name": "boolean_col",
    "mode": "NULLABLE",
    "type": "BOOLEAN",
    "description": "",
    "fields": []
  },
  {
    "name": "holiday_name",
    "mode": "NULLABLE",
    "type": "STRING",
    "description": "",
    "fields": []
  },
  {
    "name": "business_datetime",
    "mode": "NULLABLE",
    "type": "TIMESTAMP",
    "description": "",
    "fields": []
  },
  {
    "name": "created_datetime",
    "mode": "NULLABLE",
    "type": "TIMESTAMP",
    "description": "",
    "fields": []
  }
]

import json
with open('schema.json', 'w') as f :
    f.write(json.dumps(schema_setup, indent= 4))

## Load

In [ ]:
from google.cloud import bigquery
import duckdb

# Ensure Data Type
duckdb.query("SELECT * FROM df").write_parquet("output.parquet")

# Create Connection
client= bigquery.Client(BQ_TARGET_PROJECT)
table_full_name= f'{BQ_TARGET_PROJECT}.{BQ_TARGET_SCHEMA}.{BQ_TARGET_TABLE}'

# Create job_config
## schema
schema= client.schema_from_json('schema.json')
## partition
type_ = bigquery.TimePartitioningType.YEAR
time_partitioning= bigquery.TimePartitioning(field= 'timestamp_col', type_= type_)
## into job_config
job_config = bigquery.LoadJobConfig(
                                    schema= schema, 
                                    write_disposition="WRITE_TRUNCATE",
                                    time_partitioning= time_partitioning,
                                    source_format = bigquery.SourceFormat.PARQUET
                                    )

with open('output.parquet', "rb") as source_file:
    # Run a load job, with job_config
    job = client.load_table_from_file(   source_file, 
                                        f"{table_full_name}",
                                        job_config=job_config)
# Wait until the job finish
job.result()

# Clean up

In [ ]:
to_remove= ['schema.json','output.parquet',LOCAL_PATH]
for i in to_remove : os.remove(i)